# Continuous Optimisation

## Imports

In [2]:
import numpy as np
from numpy import linalg as LA
from typing import Tuple, Callable, Optional


## Medical Inverse Problem

In [4]:
class MedicalModel:
    def __init__(self, A: np.ndarray, y: np.ndarray):
        self.A = np.asarray(A, dtype=float).copy()
        self.y = np.asarray(y, dtype=float).copy()

        self.m, self.n = self.A.shape
        assert self.y.shape == (self.m,)

    def _model(self, theta: np.ndarray) -> np.ndarray:
        return self.A @ np.exp(-theta)

    def _objective(self, theta: np.ndarray) -> float:
        """Compute the least-squares objective."""
        with np.errstate(over="ignore", invalid="ignore"):
            return float(LA.norm(self._model(theta) - self.y)**2)

    def _gradient(self, theta: np.ndarray) -> np.ndarray:
        """Compute the gradient of the objective."""
        D = np.diag(np.exp(-theta))
        return -2 * D @ self.A.T @ (self._model(theta) - self.y)

    def _hessian(self, theta: np.ndarray) -> np.ndarray:
        """Compute the Hessian of the objective."""
        D = np.diag(np.exp(-theta))
        AD = self.A @ D

        return 2 * (
            np.diag(D @ self.A.T @ (self._model(theta) - self.y))
            + AD.T @ AD
        )

    def gradient_descent(
            self, 
            u_0: np.ndarray, 
            alpha: float, 
            epsilon: float = 1e-8, 
            max_iter: int = 10000
            ) -> Tuple[np.ndarray, int, float]:
        """Gradient Descent with Fixed Step Size
        
        Args:
            u_0 (np.ndarray): initial point
            alpha (float): step size
            epsilon (float, optional): error margin. Defaults to 1e-8.
            max_iter (int, optional): maximum iterations (for debugging). Defaults to 10000.

        Raises:
            ValueError: Objective function value became infinite.
            RuntimeError: Way too many iterations for the algorithm.

        Returns:
            Tuple[np.ndarray, int, float]: Returns current position, current step number, and current objective value.
        """ 
        assert 0 < alpha, f"alpha = {alpha} is not > 0."
        assert 0 < epsilon, f"epsilon = {epsilon} is not > 0."
        u_k = np.asarray(u_0, dtype=float).copy()

        for k in range(max_iter):
            grad_k = self._gradient(u_k)
            grad_norm = LA.norm(grad_k)
            
            if grad_norm < epsilon:
                return u_k, k, self._objective(u_k)

            u_k -= alpha * grad_k
            
            if not np.isfinite(self._objective(u_k)):
                raise ValueError("Objective became non-finite")
    
        raise RuntimeError(f"Did not converge within {max_iter} iterations")


    def armijo_condition(self,
        u: np.ndarray,
        d: np.ndarray,
        alpha: float,
        c: float
        ) -> bool:
        """Check E(u + alpha * d) <= E(u) + c* alpha * grad E(u)^T * d.

        Args:
            d (np.ndarray): descent direction
            c (float): constant coefficient
            alpha (float): step size

        Returns:
            bool: Is the Armijo condition satisfied.
        """        
        assert (0 < c and c < 1), f"c = {c} is not in (0, 1)."
        assert 0 < alpha, f"alpha = {alpha} is not > 0."
        step = self._objective(u + alpha * d)
        current = self._objective(u)
        change = c * alpha * self._gradient(u).T @ d
        return bool(np.isfinite(step) and step <= current + change)  

    def backtracking(self,
        u_k: np.ndarray,
        d_k: np.ndarray,
        alpha_0: float = 1.0,
        rho: float = 0.5, 
        c: float = 1e-4,
        max_iter: int = 10000
        ) -> float:
        """backtracking line search for sufficiently optimal alpha

        Args:
            u_k (np.ndarray): current point
            d_k (np.ndarray): descent direcion
            alpha_0 (float, optional): initial alpha. Defaults to 1.0.
            rho (float, optional): constant rho. Defaults to 0.5.
            c (float, optional): constant c. Defaults to 1e-4.
            max_iter (int, optional): maximum iterations (for debugging). Defaults to 10000.

        Returns:
            float: new sufficient alpha
        """        
        assert (0 < c and c < 1), f"c = {c} is not in (0, 1)."
        assert 0 < alpha_0, f"alpha = {alpha_0} is not > 0."
        assert (0 < rho and rho < 1), f"rho = {rho} is not in (0, 1)."
        alpha_new = alpha_0
        for _ in range(max_iter):
            
            if self.armijo_condition(u_k, d_k, alpha_new, c):
                return float(alpha_new)
            
            alpha_new *= rho

        raise RuntimeError("Backtracking failed to find an Armijo step.")

    def GD_inexact_line_search(self,
        u_0: np.ndarray,
        alpha_0: float = 1.0,
        rho: float = 0.5,
        c: float = 1e-4,
        epsilon: float = 1e-8,
        max_iter: int = 10000
        ) -> Tuple[np.ndarray, int, float]:
        """Gradient Descent with Inexact Line Search

        Args:
            u_0 (np.ndarray): initial point
            alpha_0 (float, optional): initial alpha. Defaults to 1.0.
            rho (float, optional): constant rho. Defaults to 0.5.
            c (float, optional): constant c. Defaults to 1e-4.
            epsilon (float, optional): error margin. Defaults to 1e-8.
            max_iter (int, optional): maximum iterations (for debugging). Defaults to 10000.

        Raises:
            ValueError: Objective function value became infinite.
            RuntimeError: Way too many iterations for the algorithm.

        Returns:
            Tuple[np.ndarray, int, float]: Returns current position, current step number, and current objective value.
        """        
        assert 0 < epsilon, f"epsilon = {epsilon} is not > 0."
        assert (0 < c and c < 1), f"c = {c} is not in (0, 1)."
        assert 0 < alpha_0, f"alpha = {alpha_0} is not > 0."
        assert (0 < rho and rho < 1), f"rho = {rho} is not in (0, 1)."
        
        u_k = np.asarray(u_0, dtype=float).copy()
        alpha_k = alpha_0

        for k in range(max_iter):
            grad_k = self._gradient(u_k) # = d_k
            grad_norm = LA.norm(grad_k)
            
            if grad_norm < epsilon:
                return u_k, k, self._objective(u_k)

            alpha_k = self.backtracking(u_k=u_k,d_k=(-grad_k),alpha_0=alpha_0,rho=rho,c=c)
            u_k -= alpha_k * grad_k
            
            if not np.isfinite(self._objective(u_k)):
                raise ValueError("Objective became non-finite")
    
        raise RuntimeError(f"Did not converge within {max_iter} iterations")

### Datasets

In [12]:
A1 = np.array([
    [1.00, 0.20, 0.10],
    [0.10, 1.00, 0.20],
    [0.20, 0.10, 1.00],
    [0.80, 0.30, 0.20],
    [0.20, 0.80, 0.30]])

y1 = np.array([0.75127762, 0.61974436,0.71505066,0.72978554,0.64541279])

A2 = np.array([
    [1.00, 0.95, 0.10],
    [0.10, 0.95, 1.00],
    [0.80, 0.76, 0.20],
    [0.20, 0.19, 0.80],
    [0.60, 0.57, 0.30]])

y2 = np.array([1.08827434,1.03632722,0.93647687,0.64572794,0.78467940])

theta_true = np.array([0.5, 0.8, 0.6])

models = [MedicalModel(A1, y1), MedicalModel(A2, y2)]

### Numerical gradient verification 

In [13]:
#Checking the gradient at several points. Selected such that they are away from the expected minimiser
test_points = [np.zeros(3),np.ones(3),np.array([-0.86, 4.04, 0.37])]

for number, model in enumerate(models, start=1):
    print(f"\nDataset {number}")

    for theta in test_points:
        print(f"\nTest point: {theta}")
        #Gradient obtained from our derived formula.
        grad = model._gradient(theta)
        #Varying h in order to examine the accuracy of finite-difference approximation.
        for h in [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]:
            grad_numeric = np.zeros(model.n)

            for i in range(model.n):
                e_i = np.zeros(model.n)
                e_i[i] = 1.0
                #Approximating the ith partial derivative using central differences
                grad_numeric[i] = (model._objective(theta + h * e_i) - model._objective(theta - h * e_i)) / (2 * h)

            Obtained_error = LA.norm(grad_numeric - grad)
            scaled_error = Obtained_error / max(1.0, LA.norm(grad))

            print(f"h = {h:.0e}, scaled error = {scaled_error:.3e}")

            if h == 1e-5:
                assert scaled_error < 1e-7


Dataset 1

Test point: [0. 0. 0.]
h = 1e-02, scaled error = 7.629e-05
h = 1e-03, scaled error = 7.629e-07
h = 1e-04, scaled error = 7.629e-09
h = 1e-05, scaled error = 7.458e-11
h = 1e-06, scaled error = 6.881e-11

Test point: [1. 1. 1.]
h = 1e-02, scaled error = 2.737e-05
h = 1e-03, scaled error = 2.737e-07
h = 1e-04, scaled error = 2.737e-09
h = 1e-05, scaled error = 2.686e-11
h = 1e-06, scaled error = 7.639e-11

Test point: [-0.86  4.04  0.37]
h = 1e-02, scaled error = 8.938e-05
h = 1e-03, scaled error = 8.938e-07
h = 1e-04, scaled error = 8.938e-09
h = 1e-05, scaled error = 9.419e-11
h = 1e-06, scaled error = 3.589e-11

Dataset 2

Test point: [0. 0. 0.]
h = 1e-02, scaled error = 6.257e-05
h = 1e-03, scaled error = 6.257e-07
h = 1e-04, scaled error = 6.257e-09
h = 1e-05, scaled error = 6.102e-11
h = 1e-06, scaled error = 7.741e-11

Test point: [1. 1. 1.]
h = 1e-02, scaled error = 3.414e-05
h = 1e-03, scaled error = 3.414e-07
h = 1e-04, scaled error = 3.414e-09
h = 1e-05, scaled err

### Results